In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import json
from pathlib import Path

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from utils.tools import aggregate_results

In [3]:
def get_suite(row):

    n_demos = row["number of demonstrations"]
    type_demos = row["type of demonstrations"][0:3]
    instr = "impl" if row["use instructions"] == "no" else "expl"

    return f"{n_demos}-{type_demos}-{instr}"

def capitalize(s):
    return s[0].upper() + s[1:]

def order_suites(strings):
    return sorted(strings, key=lambda s: (
        s.split('-')[2],  # implicit-excplict
        s.split('-')[0],  # num demos
        s.split('-')[1],  # demo type
    ))

def order_models(strings):
    return sorted(strings, key=lambda s: (
        int(s.split('-')[1][:-1]),  # Param count, B dropped
        #s.split('-')[0],  # Model name
    ), reverse=True)

def bold_extreme_values(s, by_model=True):
    # Bold max for mean

    if not by_model:
        is_max = s == s.max()
        return ['font-weight: bold' if v else '' for v in is_max]
    
    font_array = []

    model_level = s.index.names.index('model')
    models = s.index.get_level_values(model_level)
    models = pd.Series(list(models)).unique()

    idx = pd.IndexSlice
    
    for model in models:   
        values_by_model = s.loc[idx[model]]
        is_max = values_by_model == values_by_model.max()
        font_array += ['font-weight: bold' if v else '' for v in is_max]

    return font_array

In [4]:
def sig3(x):
    if isinstance(x, (int, float, np.floating)):
        return f"{x:.3g}"
    return x

def highlight_extrema(s):
    styles = []
    model_level = s.index.names.index('Model')
    models = s.index.get_level_values(model_level)
    models = pd.Series(list(models)).unique()
    
    for model in models:   
        sm = s.loc[model]
        for v in sm:
            style = ""
            if v == sm.max():
                style = 'font-weight: bold;'
            if v == sm.min():
                style = 'font-weight: bold; font-style: italic;'
            styles.append(style)
    
    return styles

In [5]:
res = pd.read_csv("./data/metrics_v3.csv", sep=";")
#res = res.set_index(['model', 'number of demonstrations', 'type of demonstrations', 'use instructions'])
#res = res.sort_values(by=["use instructions", "number of demonstrations", "type of demonstrations"])

res["suite"] = res.apply(get_suite, axis=1)
res = res.set_index(["model", "suite"])

In [6]:
std_cols = [c for c in res.columns if "std" in c and "total" not in c]
std_df = res[std_cols].copy()
std_df.columns = [" ".join([capitalize(p) for p in col.split(" ")]) for col in std_df.columns]

std_df.columns = pd.MultiIndex.from_tuples(
    [(p[0], p[1]) for p in std_df.columns.str.split(' ')],
    names=["Label", "Metric"]
)
std_df.index.names = ["Model", "Suite"]
latex = (
    std_df.style
        .apply(highlight_extrema, axis=0)
        .format(sig3)
        .to_latex(
            column_format="ll|rrrr|rrrr|rrrr",
            hrules=True,
            convert_css=True,
            multicol_align="l",
            clines="skip-last;data"
        )
)

print(latex)

\begin{tabular}{ll|rrrr|rrrr|rrrr}
\toprule
 & Label & \multicolumn{4}{l}{Theme} & \multicolumn{4}{l}{Topic} & \multicolumn{4}{l}{Concept} \\
 & Metric & Accuracy & Precision & Recall & F1 & Accuracy & Precision & Recall & F1 & Accuracy & Precision & Recall & F1 \\
Model & Suite &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
\multirow[c]{11}{*}{Llama-8B} & 0-non-expl & \bfseries \itshape 0.00505 & \bfseries \itshape 0.00366 & \bfseries \itshape 0.0104 & \bfseries \itshape 0.00846 & \bfseries \itshape 0.00626 & 0.00526 & 0.00723 & \bfseries \itshape 0.0055 & \bfseries \itshape 0.00863 & \bfseries \itshape 0.00651 & \bfseries \itshape 0.00699 & \bfseries \itshape 0.00615 \\
 & 1-neg-impl & 0.0504 & 0.0402 & 0.0964 & 0.0245 & 0.0459 & 0.0432 & 0.0933 & 0.0251 & 0.0446 & 0.17 & \bfseries 0.211 & 0.056 \\
 & 1-neg-expl & \bfseries 0.0607 & 0.0519 & 0.0252 & 0.036 & 0.0354 & 0.0202 & \bfseries \itshape 0.00462 & 0.0163 & 0.049 & 0.0375 & 0.0173 & 0.0239 \\
 & 1-pos-impl & 0.0381 & 0.0177 & 

In [7]:
score_cols = [c for c in res.columns if "mean" in c and "total" not in c]
means = res[score_cols].copy()

means.columns = [" ".join([capitalize(p) for p in col.split(" ")]) for col in means.columns]

parts = means.columns.str.split(' ')
means.columns = pd.MultiIndex.from_tuples(
    [(p[0], p[1]) for p in parts],
    names=["Label", "Metric"]
)
means.index.names = ["Model", "Suite"]

latex_mu = (
    means.style
        .apply(highlight_extrema, axis=0)
        .format(sig3)
        .to_latex(
            column_format="ll|rrrr|rrrr|rrrr",
            hrules=True,
            convert_css=True,
            multicol_align="l",
            clines="skip-last;data"
        )
)

print(latex_mu)

\begin{tabular}{ll|rrrr|rrrr|rrrr}
\toprule
 & Label & \multicolumn{4}{l}{Theme} & \multicolumn{4}{l}{Topic} & \multicolumn{4}{l}{Concept} \\
 & Metric & Accuracy & Precision & Recall & F1 & Accuracy & Precision & Recall & F1 & Accuracy & Precision & Recall & F1 \\
Model & Suite &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
\multirow[c]{11}{*}{Llama-8B} & 0-non-expl & 0.771 & 0.913 & 0.563 & 0.697 & 0.807 & 0.738 & 0.948 & 0.83 & 0.683 & 0.632 & 0.919 & 0.749 \\
 & 1-neg-impl & \bfseries \itshape 0.603 & \bfseries \itshape 0.55 & 0.894 & 0.677 & 0.661 & 0.612 & 0.902 & 0.725 & 0.712 & 0.807 & 0.679 & 0.699 \\
 & 1-neg-expl & 0.664 & 0.592 & \bfseries 0.929 & 0.722 & \bfseries \itshape 0.548 & \bfseries \itshape 0.524 & \bfseries 0.992 & 0.686 & \bfseries \itshape 0.651 & \bfseries \itshape 0.601 & \bfseries 0.971 & 0.741 \\
 & 1-pos-impl & 0.627 & 0.887 & \bfseries \itshape 0.299 & \bfseries \itshape 0.444 & 0.685 & \bfseries 1 & 0.419 & 0.588 & 0.658 & 0.812 & \bfseries \itshape 0.5